# Assignment 1 — EDA and Baseline Models
**CENG313 · GROUP25 · Kaggle: [WSDM Cup – Multilingual Chatbot Arena](https://www.kaggle.com/competitions/wsdm-cup-multilingual-chatbot-arena)**

Users of Chatbot Arena send a prompt to two anonymous LLMs and vote for the answer they prefer.
Given `prompt`, `response_a`, `response_b` we predict the `winner` (`model_a` / `model_b`). Metric: **accuracy**.

Contents
1. Setup & data loading
2. Exploratory data analysis
3. Baseline models (5-fold CV)
4. Error analysis
5. Baseline submission
6. Conclusions

All reusable code (feature extraction, models, CV, plots) lives in the `GROUP25_CENG313/` package of the repository; this notebook only calls it.

## 1. Setup & data loading

In [ ]:
# ---- Environment setup (works on Google Colab and locally) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/shroudy-bit32/GROUP25_CENG313.git"   # <-- set to your repo
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("GROUP25_CENG313").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("GROUP25_CENG313")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # running locally from the notebooks/ folder -> go to repo root
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

In [ ]:
# ---- Get the competition data ----
# Option 1 (Colab): Kaggle API. Upload kaggle.json (Kaggle -> Settings -> API -> Create New Token).
# Option 2 (Colab): put the parquet files in Google Drive: MyDrive/wsdm-cup-multilingual-chatbot-arena/
# Option 3 (local): put train.parquet / test.parquet into ./data
from pathlib import Path

if IN_COLAB and not Path("data/train.parquet").exists():
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    else:
        from google.colab import files
        if not Path("~/.kaggle/kaggle.json").expanduser().exists():
            print("Upload kaggle.json")
            files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        subprocess.run("kaggle competitions download -c wsdm-cup-multilingual-chatbot-arena -p data "
                       "&& cd data && unzip -o -q wsdm-cup-multilingual-chatbot-arena.zip", shell=True, check=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110

from GROUP25_CENG313 import data as D, eda as E
from GROUP25_CENG313.features import HandcraftedFeatureExtractor

SEED = 42
train, test = D.load_competition_data()
print("train:", train.shape, " test:", test.shape)
train.head(3)

In [ ]:
E.basic_overview(train)

In [ ]:
print("Test columns:", test.columns.tolist())
test.head()

## 2. Exploratory data analysis
### 2.1 Target distribution & position bias
If labels were perfectly balanced, P(A wins) would be 0.5. Any deviation indicates a **position bias** of the voters (or of the data collection).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
E.plot_target_distribution(train, ax=ax)
plt.show()
print(train["winner"].value_counts(normalize=True).round(4))

### 2.2 Languages

In [ ]:
print("Number of languages:", train["language"].nunique())
fig, ax = plt.subplots(figsize=(7, 6))
E.plot_top_languages(train, top=20, ax=ax)
plt.tight_layout(); plt.show()
lang_share = train["language"].value_counts(normalize=True)
print(f"Top-1 language share: {lang_share.iloc[0]:.1%} ({lang_share.index[0]}); "
      f"top-5 share: {lang_share.iloc[:5].sum():.1%}")

In [ ]:
# Does the A/B win rate change with language?
lang_wr = (train.assign(a_wins=train["winner"].eq("model_a"))
                .groupby("language")["a_wins"].agg(["mean", "size"])
                .query("size >= 300").sort_values("size", ascending=False)
                .rename(columns={"mean": "P(A wins)", "size": "n"}))
lang_wr.head(15)

### 2.3 Text lengths & verbosity bias

In [ ]:
for c in ["prompt", "response_a", "response_b"]:
    s = train[c].str.len()
    print(f"{c:11s} chars  median={s.median():7.0f}  mean={s.mean():7.0f}  p95={s.quantile(.95):7.0f}  max={s.max():8.0f}")
fig, ax = plt.subplots(figsize=(8, 4))
E.plot_length_distributions(train, ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
E.plot_verbosity_bias(train, n_bins=10, ax=ax)
plt.tight_layout(); plt.show()
E.verbosity_bias_table(train)

In [ ]:
la, lb = train["response_a"].str.len(), train["response_b"].str.len()
longer_wins = ((la > lb) & train.winner.eq("model_a")) | ((lb > la) & train.winner.eq("model_b"))
print(f"The longer response wins in {longer_wins[la != lb].mean():.2%} of battles (ties in length excluded)")

### 2.4 Formatting (markdown) features
Do voters prefer structured answers (headers, lists, bold text)? We compare P(A wins) when A has more / the same / fewer formatting elements than B.

In [ ]:
import re
patterns = {
    "headers (#)":  r"(?m)^\s{0,3}#{1,6}\s",
    "bullet lists": r"(?m)^\s*[-*+•]\s+",
    "numbered lists": r"(?m)^\s*\d+[.)]\s+",
    "bold (**)":    r"\*\*[^*]+\*\*",
    "code blocks":  r"```",
}
tables = {}
for name, pat in patterns.items():
    fa = train["response_a"].str.count(pat)
    fb = train["response_b"].str.count(pat)
    tables[name] = E.feature_win_rate(train, fa, fb)["P(A wins)"]
pd.DataFrame(tables).T[["A > B", "A = B", "A < B"]].round(3)

### 2.5 Which LLMs take part and how often do they win?

In [ ]:
print("Distinct models:", pd.concat([train.model_a, train.model_b]).nunique())
wr = E.model_win_rates(train, min_battles=500)
fig, ax = plt.subplots(figsize=(7, 9))
wr["win_rate"].head(30)[::-1].plot.barh(ax=ax, color="#5b8c5a")
ax.axvline(0.5, ls="--", color="gray"); ax.set_title("Win rate of the 30 best models (>=500 battles)")
plt.tight_layout(); plt.show()
wr.head(10)

*Note:* model names are **not** available in the test set, so they cannot be used as features directly. They are still useful to understand the data (e.g. strong models write longer, better formatted answers).

### 2.6 Special cases: empty, identical and refusal responses

In [ ]:
stats = {}
stats["identical responses"] = (train.response_a.str.strip() == train.response_b.str.strip()).mean()
stats["A or B empty"] = ((train.response_a.str.len() == 0) | (train.response_b.str.len() == 0)).mean()
refusal = train.response_a.str.slice(0, 300).str.contains(
    r"I'm sorry|I am sorry|I can't|I cannot|as an AI|lo siento|не могу|抱歉|无法", case=False, regex=True)
stats["A looks like a refusal"] = refusal.mean()
print(pd.Series(stats).map("{:.2%}".format))
print(f"P(A wins | A is a refusal) = {train.loc[refusal, 'winner'].eq('model_a').mean():.3f}")

In [ ]:
# One random example
ex = train.sample(1, random_state=3).iloc[0]
print("LANGUAGE:", ex.language, "| WINNER:", ex.winner, "|", ex.model_a, "vs", ex.model_b)
print("\nPROMPT:\n", ex.prompt[:800])
print("\nRESPONSE A:\n", ex.response_a[:800])
print("\nRESPONSE B:\n", ex.response_b[:800])

### 2.7 EDA summary
* **48,439** labelled battles, **128 languages** (English 52%, Russian 13%, Chinese 9%, Vietnamese 6%; top-5 = 84%), 60 different LLMs, no missing values.
* **Balanced target:** model_b wins 50.5% → a tiny position bias towards the second answer; the majority class gives only 50.5% accuracy.
* **Verbosity bias is the strongest signal:** the longer answer wins **58.4%** of the time; when A is much longer than B (top decile of the length ratio) A wins ~62%, when it is much shorter only ~36%.
* **Formatting bias:** answers with more headers / bold text / lists / code blocks win more often (e.g. P(A wins) = 0.61 if A has more headers vs 0.37 if fewer).
* **Refusals lose:** when A looks like a refusal it wins only 26% of the time.
* Strong models (chatgpt-4o-latest, o1-preview, gemini-1.5-pro-exp) win ~67–72% of their battles, but model names are **not available in the test set**.
* Prompts are short (median 142 chars) while answers are long (median ~1,600 chars, max >100k) → transformer models in Assignment 2 will need truncation strategies.